# Module 3.0: AI Search Index Setup

In upcoming notebooks, agents will need to answer policy questions like
*"What's the hotel budget for a Senior IC?"* or *"Which vendors are preferred?"*

Without a ground-truth source, agents **hallucinate** policy details — inventing
budget limits, making up vendor preferences, and confidently citing rules that
don't exist. This notebook builds the authoritative policy index that prevents that.

> **The question this notebook answers:**
> How do we give the agent a reliable, searchable source of truth for corporate policy?

In [ ]:
%pip install -q azure-search-documents azure-identity openai

In [ ]:
import sys, os, json
from pathlib import Path
from datetime import datetime, timezone

sys.path.insert(0, "..")

from dotenv import load_dotenv
load_dotenv("../.env")

from azure.identity import AzureCliCredential

credential = AzureCliCredential()

SEARCH_ENDPOINT = os.environ["AZURE_SEARCH_ENDPOINT"]
EMBEDDING_MODEL = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-small")
FOUNDRY_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]

print(f"Search endpoint: {SEARCH_ENDPOINT}")
print(f"Embedding model: {EMBEDDING_MODEL}")

## The Problem: Agents Hallucinate Policy

Ask an LLM a policy question without grounding and it confidently invents an answer.
Watch what happens when we ask about hotel budgets with **no policy source**:

In [ ]:
import sniffio
sniffio.current_async_library_cvar.set("asyncio")

from shared.travel_agent import create_client
from agent_framework._types import Message

client, credential_chat = create_client("../.env")

# Ask a policy question with NO grounding — pure hallucination
messages = [
    Message(role="system", contents=["You are a corporate travel assistant."]),
    Message(role="user", contents=["What is the maximum hotel budget for a Senior IC employee?"]),
]

response = await client.get_response(messages=messages)
print("Agent (no policy source):")
print(f"  {response.text}")
print()
print("⚠️  Is that number correct? We have no way to know.")
print("   The agent invented a plausible-sounding answer with no source.")

## What Went Wrong

The agent has no access to actual corporate policy documents. It fills the gap with
plausible-sounding but **fabricated** numbers — a hallucination that could lead to
over-budget bookings, policy violations, or compliance failures.

| Without Ground Truth | With Ground Truth (AI Search) |
|---------------------|-------------------------------|
| Invents budget numbers | Retrieves actual policy limits |
| Makes up vendor preferences | Returns contracted vendor list |
| Guesses per-diem rates | Looks up city-specific rates |
| No audit trail | Every answer traceable to a document + version |

## The Solution: AI Search as RAG Ground Truth

We index all corporate policy documents into Azure AI Search with vector embeddings.
This gives agents **hybrid search** (keyword + semantic) over authoritative content.

When memory contradicts RAG, memory is wrong (→ Notebook 07: Procedural Lifecycle).

### Prerequisites

1. Azure AI Search resource (Free tier works)
2. Embedding model in Azure AI Foundry (e.g. `text-embedding-ada-002`)
3. Environment variables: `AZURE_SEARCH_ENDPOINT`, `AZURE_SEARCH_KEY`, `FOUNDRY_PROJECT_ENDPOINT`

## Building the Index: Schema Definition

Each document in the index has:
- `id` — unique document identifier
- `title` — document title for display
- `content` — full text (searchable)
- `category` — document type for filtering
- `version` — version string for change detection (used in Notebook 07)
- `last_updated` — when the document was last modified
- `content_vector` — embedding for semantic search

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    SemanticConfiguration,
    SemanticSearch,
    SemanticPrioritizedFields,
    SemanticField,
    SearchField,
)
from azure.core.credentials import AzureKeyCredential

INDEX_NAME = "travel-policies"
VECTOR_DIMENSIONS = 1536  # text-embedding-ada-002

SEARCH_KEY = os.environ["AZURE_SEARCH_KEY"]

index_client = SearchIndexClient(
    endpoint=SEARCH_ENDPOINT,
    credential=AzureKeyCredential(SEARCH_KEY),
)

# Define fields
fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True, filterable=True),
    SearchableField(name="title", type=SearchFieldDataType.String, filterable=True),
    SearchableField(name="content", type=SearchFieldDataType.String),
    SimpleField(name="category", type=SearchFieldDataType.String, filterable=True, facetable=True),
    SimpleField(name="version", type=SearchFieldDataType.String, filterable=True),
    SimpleField(name="last_updated", type=SearchFieldDataType.DateTimeOffset, filterable=True, sortable=True),
    SearchField(
        name="content_vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=VECTOR_DIMENSIONS,
        vector_search_profile_name="default-profile",
    ),
]

# Vector search configuration
vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="default-algorithm")],
    profiles=[VectorSearchProfile(name="default-profile", algorithm_configuration_name="default-algorithm")],
)

# Semantic configuration
semantic_config = SemanticConfiguration(
    name="default-semantic",
    prioritized_fields=SemanticPrioritizedFields(
        title_field=SemanticField(field_name="title"),
        content_fields=[SemanticField(field_name="content")],
    ),
)
semantic_search = SemanticSearch(configurations=[semantic_config])

# Create or update the index
index = SearchIndex(
    name=INDEX_NAME,
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search,
)

result = index_client.create_or_update_index(index)
print(f"Index '{result.name}' created/updated successfully")
print(f"  Fields: {len(result.fields)}")
print(f"  Vector search: {result.vector_search is not None}")
print(f"  Semantic search: {result.semantic_search is not None}")

## Preparing Documents

We load all policy documents from `data/policies/` and existing skill files,
then structure them for upload with metadata for version tracking.

In [ ]:
POLICIES_DIR = Path("../data/policies")
SKILLS_DIR = Path("../02_memory_layers/skills")


def load_document(filepath: Path, doc_id: str, title: str, category: str,
                  version: str = "1.0") -> dict:
    """Load a file and structure it as a search document."""
    content = filepath.read_text(encoding="utf-8")
    # For JSON files, convert to readable text
    if filepath.suffix == ".json":
        data = json.loads(content)
        content = json.dumps(data, indent=2)
    return {
        "id": doc_id,
        "title": title,
        "content": content,
        "category": category,
        "version": version,
        "last_updated": datetime.now(timezone.utc).isoformat(),
    }


# Collect all documents
documents = [
    # Policy documents
    load_document(
        POLICIES_DIR / "general_travel_policy.md",
        "general-travel-policy",
        "General Travel Policy",
        "policy",
        version="4.2",
    ),
    load_document(
        POLICIES_DIR / "expense_reimbursement.md",
        "expense-reimbursement",
        "Expense Reimbursement Policy",
        "policy",
        version="2.1",
    ),
    load_document(
        POLICIES_DIR / "preferred_vendors.md",
        "preferred-vendors",
        "Preferred Vendors Policy",
        "policy",
        version="3.0",
    ),
    load_document(
        POLICIES_DIR / "travel_safety.md",
        "travel-safety",
        "Travel Safety & Compliance Policy",
        "compliance",
        version="1.4",
    ),
    load_document(
        POLICIES_DIR / "per_diem_rates.json",
        "per-diem-rates",
        "Per-Diem Rates by City",
        "policy",
        version="2026-Q3",
    ),
    # Existing skill/procedure documents
    load_document(
        SKILLS_DIR / "domestic-booking" / "SKILL.md",
        "domestic-booking-procedure",
        "Domestic Booking Procedure",
        "procedure",
    ),
    load_document(
        SKILLS_DIR / "international-booking" / "SKILL.md",
        "international-booking-procedure",
        "International Booking Procedure",
        "procedure",
    ),
    load_document(
        SKILLS_DIR / "international-booking" / "visa-checklist.md",
        "visa-checklist",
        "Visa Requirements Checklist",
        "compliance",
    ),
    load_document(
        SKILLS_DIR / "domestic-booking" / "budget-limits.json",
        "budget-limits-domestic",
        "Budget Limits — Domestic Travel",
        "policy",
    ),
    load_document(
        SKILLS_DIR / "international-booking" / "budget-limits.json",
        "budget-limits-international",
        "Budget Limits — International Travel",
        "policy",
    ),
]

print(f"Prepared {len(documents)} documents for indexing:")
print()
for doc in documents:
    print(f"  [{doc['category']:<10}] {doc['id']:<35} v{doc['version']}")

## Generating Embeddings

We generate vector embeddings using the Foundry **account-level** endpoint.

> **Note:** The project-level endpoint (`/api/projects/...`) returns 404 for embeddings.
> We strip the project path to get the account endpoint, which serves `/openai/v1/embeddings`.

In [ ]:
from openai import OpenAI
from azure.identity import get_bearer_token_provider

# Embeddings are served at the ACCOUNT-level endpoint, not the project-level.
# Derive account endpoint by stripping /api/projects/{project} from FOUNDRY_PROJECT_ENDPOINT.
account_endpoint = FOUNDRY_ENDPOINT.split("/api/projects")[0]
token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")

openai_client = OpenAI(
    base_url=f"{account_endpoint}/openai/v1",
    api_key="placeholder",  # required by SDK but overridden by auth header
    default_headers={"Authorization": f"Bearer {token_provider()}"},
)


def get_embedding(text: str) -> list[float]:
    """Generate embedding for a text string."""
    response = openai_client.embeddings.create(
        input=text[:8000],  # Safe character limit
        model=EMBEDDING_MODEL,
    )
    return response.data[0].embedding


# Generate embeddings for all documents
print("Generating embeddings...")
for i, doc in enumerate(documents):
    doc["content_vector"] = get_embedding(doc["content"])
    print(f"  [{i+1}/{len(documents)}] {doc['id']} — {len(doc['content_vector'])} dimensions")

print(f"\nAll {len(documents)} embeddings generated")

## Uploading to the Index

Upload all documents with their embeddings to AI Search.

In [ ]:
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential

SEARCH_KEY = os.environ["AZURE_SEARCH_KEY"]

search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=AzureKeyCredential(SEARCH_KEY),
)

# Upload documents
result = search_client.upload_documents(documents=documents)

succeeded = sum(1 for r in result if r.succeeded)
failed = sum(1 for r in result if not r.succeeded)
print(f"Upload complete: {succeeded} succeeded, {failed} failed")

if failed:
    for r in result:
        if not r.succeeded:
            print(f"  FAILED: {r.key} — {r.error_message}")

## The Payoff: Grounded Answers Instead of Hallucinations

Now let's ask the same policy questions — but this time the agent can search
the index we just built. Compare these results against the hallucinated answer above.

In [ ]:
from azure.search.documents.models import VectorizedQuery


def search_policies(query: str, top_k: int = 3, category: str = None) -> list[dict]:
    """Search the policy index with hybrid (keyword + vector) search."""
    vector_query = VectorizedQuery(
        vector=get_embedding(query),
        k_nearest_neighbors=top_k,
        fields="content_vector",
    )

    filter_expr = f"category eq '{category}'" if category else None

    results = search_client.search(
        search_text=query,
        vector_queries=[vector_query],
        filter=filter_expr,
        top=top_k,
        select=["id", "title", "content", "category", "version", "last_updated"],
    )

    return [
        {
            "id": r["id"],
            "title": r["title"],
            "content": r["content"][:500],  # Truncate for display
            "category": r["category"],
            "version": r["version"],
            "score": r["@search.score"],
        }
        for r in results
    ]


print("search_policies() function ready")

In [ ]:
# Test queries
test_queries = [
    "What is the hotel budget limit for a Senior employee?",
    "Which hotel chain should I book?",
    "What is the per-diem rate for New York?",
    "Do I need a visa for international travel?",
    "What expenses can I claim for reimbursement?",
]

for query in test_queries:
    results = search_policies(query, top_k=2)
    print(f"\nQ: {query}")
    for r in results:
        print(f"  → [{r['category']}] {r['title']} (v{r['version']}, score={r['score']:.3f})")

## Production-Ready: Reusable Search Tool

This wraps the hybrid search into an `@tool`-compatible function that any agent
can use for RAG-based policy lookup.

In [ ]:
from agent_framework import tool


@tool
async def search_travel_policies(
    query: str, category: str = "", top_k: int = 3
) -> str:
    """Search corporate travel policies, procedures, and compliance documents.
    
    Use this to find current policy information about budgets, vendors,
    safety requirements, expense rules, and booking procedures.
    
    Args:
        query: Natural language question about travel policies
        category: Optional filter — 'policy', 'procedure', or 'compliance'
        top_k: Number of results to return (default 3)
    """
    results = search_policies(query, top_k=top_k, category=category or None)
    if not results:
        return "No matching policies found."
    return json.dumps(results, indent=2)


print("@tool search_travel_policies ready")
print("This tool can be added to any agent's tool list for RAG-based policy lookup.")

## Full-Document Retrieval for Version Comparison

For procedural lifecycle validation (Notebook 07), we need the **full document content**
to compare against stored memories. This function fetches complete policy documents by ID.

In [ ]:
def get_policy_by_id(doc_id: str) -> dict | None:
    """Retrieve a specific policy document by ID (for version comparison)."""
    try:
        result = search_client.get_document(key=doc_id)
        return {
            "id": result["id"],
            "title": result["title"],
            "content": result["content"],
            "version": result["version"],
            "last_updated": result["last_updated"],
        }
    except Exception:
        return None


# Test: get the preferred vendors policy
vendor_policy = get_policy_by_id("preferred-vendors")
if vendor_policy:
    print(f"Retrieved: {vendor_policy['title']}")
    print(f"Version:   {vendor_policy['version']}")
    print(f"Updated:   {vendor_policy['last_updated']}")
    print(f"Content:   {len(vendor_policy['content'])} characters")
    print(f"\nFirst 200 chars:")
    print(f"  {vendor_policy['content'][:200]}...")

## Key Takeaways

1. **Without grounding, agents hallucinate policy** — confident but fabricated budget limits, vendor preferences, and rules
2. **AI Search provides authoritative ground truth** — hybrid search (keyword + semantic) over real policy documents
3. **Version tracking enables staleness detection** — each document has a version string that Notebook 07 uses to detect when stored memories are outdated
4. **Full-document retrieval supports validation** — `get_policy_by_id()` lets the procedural validator compare stored reflections against current policy text

### Documents Indexed

| Document | Category | Version |
|----------|----------|----------|
| General Travel Policy | policy | 4.2 |
| Expense Reimbursement | policy | 2.1 |
| Preferred Vendors | policy | 3.0 |
| Travel Safety & Compliance | compliance | 1.4 |
| Per-Diem Rates | policy | 2026-Q3 |
| Domestic Booking Procedure | procedure | 1.0 |
| International Booking Procedure | procedure | 1.0 |
| Visa Checklist | compliance | 1.0 |
| Budget Limits (Domestic/International) | policy | 1.0 |

### Reusable Functions
- `search_policies(query, top_k, category)` — hybrid search for notebooks
- `search_travel_policies(query, category, top_k)` — `@tool` for agents
- `get_policy_by_id(doc_id)` — direct retrieval for version comparison

## Next: Memory vs RAG (Notebook 01)

Now that we have ground truth, the next notebook explores what RAG alone
can't do — and why agents need memory on top of it.